# Llama 3.2 3B Instruct - Hugging Face API

Este notebook demonstra como usar o modelo **meta-llama/Llama-3.2-3B-Instruct** via Hugging Face Inference API.

## 1. Instalar dependências

In [1]:
!pip install -q transformers huggingface_hub accelerate

## 2. Configurar autenticação

Você precisa de um token do Hugging Face. Obtenha em: https://huggingface.co/settings/tokens

In [1]:
import os
from dotenv import load_dotenv 
from huggingface_hub import login

# Opção 1: Definir como variável de ambiente
#os.environ["HF_TOKEN"] =

# Opção 2: Login interativo (vai pedir o token)
# login()

# Opção 3: Passar o token diretamente (não recomendado para código público)

#load_dotenv()
HF_TOKEN = os.environ.get("HF_TOKEN", None)

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("✅ Autenticado no Hugging Face!")
else:
    print("⚠️ Defina HF_TOKEN ou rode login() para se autenticar")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ Autenticado no Hugging Face!


## 3. Usar via Inference API (modo mais simples)

In [6]:
from huggingface_hub import InferenceClient

client = InferenceClient(token=HF_TOKEN)

# Chat básico
response = client.chat.completions.create(
    model="meta-llama/Llama-3.1-8B-Instruct",
    messages=[
        {"role": "user", "content": "Explique o que é Deep Learning em 3 frases."}
    ]
)

print(response.choices[0].message.content)

O Deep Learning (Aprendizado Profundo) é uma subárea da Inteligência Artificial que visa criar modelos neurais artificialmente inspirados na estrutura e no funcionamento do cérebro humano. Os modelos de Deep Learning usam uma camada de neurônios artificiais, em uma hierarquia de camadas, para identificar padrões e aprender a partir dos dados. Isso permite a criação de sistemas que podem reconhecer imagens, processar linguagem natural e realizar outras tarefas complexas com grande precisão e eficiência.


## 4. Chat com contexto (múltiplas mensagens)

In [8]:
messages = [
    {"role": "system", "content": "Você é um assistente especializado em IA. Seja conciso e técnico."},
    {"role": "user", "content": "Qual a diferença entre RNN e Transformer?"},
]

response = client.chat.completions.create(
    model="meta-llama/Llama-3.1-8B-Instruct",
    messages=messages
)

print(response.choices[0].message.content)

As redes neurais recorrentes (RNN) e as redes neurais de transformação (Transformer) são ambos modelos de inteligência artificial usados para processamento de linguagem natural, mas são projetados para lidar com problemas diferentes de maneira distintas.

**Redes Neurais Recorrentes (RNN):**

- RNN são projetadas para aprender dependências de sequência longa em dados sequenciais, como linguagem natural.
- Eles usam uma estrutura de feedback nas suas conexões recorrentes para capturar características de sequências.
- A comunicação é linear e serial.

**Redes Neurais de Transformação (Transformer):**

- As Transformer são projetadas para resolver problemas de linguagem natural, como tradução de texto, em que a sequência de entrada é longa e o modelo precisa aprender padrões de longo prazo.
- Eles usam uma abordagem de processamento de atenção (attention mechanism) em vez de conexões recorrentes.
- O modelo realiza operações paralelas de multi-tarefa simultaneamente, facilitando a escala 

## 5. Streaming (resposta em tempo real)

In [9]:
print("🤖 Llama 3.1: ", end="")
for chunk in client.chat.completions.create(
    model="meta-llama/Llama-3.1-8B-Instruct",
    messages=[{"role": "user", "content": "Liste 3 aplicações práticas de LLMs."}],
    stream=True
):
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()

🤖 Llama 3.1: Aqui estão 3 aplicações práticas de LLMs (Redes Neurais de Linguagem por Recurso):

1. **assistentes virtuais de conversação**: Os LLMs são usados para criar assistentes virtuais de conversação, como Siri, Alexa e Google Assistant, que podem entender e responder a perguntas e comandos humanos de forma natural.

2. **tradução automática**: Os LLMs são usados para traduzir textos e voz em tempo real, facilitando a comunicação entre pessoas que falam diferentes línguas. Isso é útil para turistas, empresários e estudantes que precisam se comunicar em diferentes idiomas.

3. **geração de conteúdo**: Os LLMs podem gerar conteúdo, como textos, imagens e vídeos, automatizando tarefas que antes eram feitas por humanos. Essa aplicação é útil para criadores de conteúdo, como escritores, designers e editores, que podem se concentrar em outras tarefas enquanto os LLMs geram conteúdo de alta qualidade.

Essas são apenas algumas das muitas aplicações práticas de LLMs. Eles estão sendo ut

## 6. Com parâmetros avançados

In [10]:
response = client.chat.completions.create(
    model="meta-llama/Llama-3.1-8B-Instruct",
    messages=[{"role": "user", "content": "Gerar um título criativo para um artigo sobre IA."}],
    temperature=0.9,
    max_tokens=50,
    top_p=0.9
)

print(response.choices[0].message.content)

Aqui estão algumas sugestões de títulos criativos para um artigo sobre IA:

1. "O Futuro é Automático: Como a IA está mudando o mundo"
2. "Código de Vida: A Rev


## 7. Usar via API REST direta

In [21]:
import requests

API_URL = "https://router.huggingface.co/v1/chat/completions"

headers = {"Authorization": f"Bearer {HF_TOKEN}"}
payload = {
    "model": "meta-llama/Llama-3.1-8B-Instruct",
    "messages": [{"role": "user", "content": "Diga olá em português."}],
}

response = requests.post(API_URL, headers=headers, json=payload)
result = response.json()

print(result)
print(result["choices"][0]["message"]["content"])

{'id': 'chatcmpl-b405afe5-fc70-493f-8e04-c965d4748747', 'choices': [{'finish_reason': 'stop', 'index': 0, 'message': {'content': 'Olá!', 'role': 'assistant'}}], 'created': 1773432290, 'model': 'llama3.1-8b', 'system_fingerprint': 'fp_5198798116a66ebf301b', 'object': 'chat.completion', 'usage': {'total_tokens': 48, 'completion_tokens': 4, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0, 'reasoning_tokens': 0}, 'prompt_tokens': 44, 'prompt_tokens_details': {'cached_tokens': 0}}, 'time_info': {'queue_time': 7.881e-05, 'prompt_time': 0.002873262, 'completion_time': 0.001602752, 'total_time': 0.005729198455810547, 'created': 1773432290.0784397}}
Olá!


## 8. Carregar modelo localmente (requer GPU)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Carregar tokenizer e modelo
model_name = "meta-llama/Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    token=HF_TOKEN,
    torch_dtype=torch.float16,
    device_map="auto"
)

print(f"✅ Modelo carregado: {model_name}")

## 9. Geração local com transformers

In [ ]:
# Formatar prompt no estilo Llama
def format_prompt(user_input, system_prompt=""):
    if system_prompt:
        return f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n{system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>\n{user_input}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"
    return f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n{user_input}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"

prompt = format_prompt("Explique transformers em uma frase.", "Seja técnico e conciso.")

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.7,
    do_sample=True,
    top_p=0.9
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

## 10. Pipeline simplificado

In [ ]:
from transformers import pipeline

# Criar pipeline de text generation
pipe = pipeline(
    "text-generation",
    model="meta-llama/Llama-3.1-8B-Instruct",
    token=HF_TOKEN,
    torch_dtype=torch.float16,
    device_map="auto"
)

messages = [{"role": "user", "content": "Qual a capital do Brasil?"}]
outputs = pipe(messages, max_new_tokens=50)
print(outputs[0]["generated_text"][-1]["content"])

## Notas

- **Inference API**: Mais simples, não requer GPU, mas tem limites de rate
- **Modelo local**: Mais controle, sem limites, mas requer GPU (pelo menos 8GB VRAM para 3B)
- **Token**: Necessário aceitar os termos do modelo em https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct